In [ ]:
# Cell 1 — 공통 실행 환경 설정
# 이 Cell은 모든 tutorial Notebook의 첫 번째 Cell이며 여러 번 실행해도 안전합니다.

from pathlib import Path
import importlib.util
import os
import subprocess
import sys

# Google Drive의 project folder가 다르면 이 값만 수정합니다.
COLAB_PROJECT_ROOT = Path(
    "/content/drive/MyDrive/DiffDodree"
)

# 필요하면 환경변수로 project root를 직접 지정할 수 있습니다.
PROJECT_ROOT_OVERRIDE = os.environ.get("GUGAKGEN_PROJECT_ROOT")


def find_local_project_root(start: Path) -> Path:
    """현재 위치에서 pyproject.toml을 기준으로 project root를 찾습니다."""
    candidates = [start.resolve(), *start.resolve().parents]

    for candidate in candidates:
        if (candidate / "pyproject.toml").exists():
            return candidate

    raise FileNotFoundError(
        "project root를 찾지 못했습니다. "
        "GUGAKGEN_PROJECT_ROOT를 지정하거나 project folder에서 Notebook을 실행하세요."
    )


try:
    # 순수 local Python에는 상위 ``google`` namespace가 없을 수 있습니다.
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    # 이미 mount되어 있으면 기존 mount를 그대로 사용합니다.
    drive.mount("/content/drive", force_remount=False)

if PROJECT_ROOT_OVERRIDE:
    project_root = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
elif IN_COLAB:
    project_root = COLAB_PROJECT_ROOT.resolve()
else:
    project_root = find_local_project_root(Path.cwd())

if not project_root.exists():
    raise FileNotFoundError(
        f"project folder가 존재하지 않습니다: {project_root}\n"
        "COLAB_PROJECT_ROOT 또는 GUGAKGEN_PROJECT_ROOT를 확인하세요."
    )

bootstrap_script = project_root / "scripts" / "bootstrap_notebook.py"

if not bootstrap_script.exists():
    raise FileNotFoundError(
        f"Notebook bootstrap script를 찾지 못했습니다: {bootstrap_script}"
    )

environment_profile = "colab" if IN_COLAB else "local"

# 필요한 dependency만 확인하고 누락된 package만 설치합니다.
# Colab profile에서는 Colab이 제공한 CUDA용 PyTorch를 교체하지 않습니다.
subprocess.run(
    [
        sys.executable,
        str(bootstrap_script),
        "--project-root",
        str(project_root),
        "--profile",
        environment_profile,
        "--install-missing",
    ],
    check=True,
)

# 현재 kernel에서 src package를 즉시 import할 수 있게 합니다.
src_root = project_root / "src"

if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

os.environ["GUGAKGEN_PROJECT_ROOT"] = str(project_root)
os.chdir(project_root)

from gugakgen.device import resolve_device
from gugakgen.reproducibility import set_global_seed

runtime = resolve_device(preference="auto")
device = runtime.device

SEED = 2026
set_global_seed(SEED)

print("환경 설정 완료")
print(f"- 실행 환경: {'Google Colab' if IN_COLAB else 'Local Jupyter'}")
print(f"- Project root: {project_root}")
print(f"- Device: {device}")
print(f"- AMP enabled: {runtime.amp_enabled}")
print(f"- Seed: {SEED}")

# 05. 환입 Counterfactual experiment

이 Notebook은 새로운 제1장 material이 제4장 환입 구간에 관계적으로 반영되는지 확인합니다.
먼저 제1장의 네 마디만 새로 생성한 뒤, 그 material을 고정한 상태에서 제4장 25–28마디를
다시 생성합니다.

비교 조건은 Proposed Graph-conditioned Model, No-return ablation, Rule Baseline,
Negative control입니다. 동일 Seed와 Sampling configuration을 사용하여 조건 간 차이를
paired comparison으로 읽습니다.

In [ ]:
from dataclasses import replace

import numpy as np
import pandas as pd
from IPython.display import display

from gugakgen.data.records import TokenizedScore
from gugakgen.structure.hwanip_relations import HwanipRelation, HwanipRelations
from gugakgen.workflows.generate import load_generation_bundle

checkpoint_candidates = [
    project_root / "artifacts/checkpoints/yangcheongdodeuri_maskgit.pt",
    project_root / "artifacts/checkpoints/yangcheongdodeuri_maskgit_smoke.pt",
    project_root / "artifacts/m2_denoiser_v0.pt",
]
checkpoint_path = next((path for path in checkpoint_candidates if path.exists()), None)
if checkpoint_path is None:
    raise FileNotFoundError("사용 가능한 Checkpoint가 없습니다.")

bundle = load_generation_bundle(checkpoint_path, project_root=project_root, runtime=runtime)
true_relations = bundle.context.hwanip_relations
N_TRIALS = 2  # tutorial 확인용. 논문 실험에서는 사전등록한 sample 수로 늘립니다.
SAMPLING_STEPS = 4
SEED_START = 3100

experiment_config = replace(
    bundle.sampling_config,
    num_steps=SAMPLING_STEPS,
    copy_rejection_attempts=2,
    hwanip_guidance_scale=1.0,
)

print("Checkpoint:", checkpoint_path.relative_to(project_root))
print("환입 pair:", [(a + 1, b + 1) for a, b in true_relations.pairs])
if "smoke" in checkpoint_path.name:
    print("주의: 현재 결과는 experiment code 확인용이며 연구 결과가 아닙니다.")

In [ ]:
# TokenizedScore에서 마디 signature와 Rule Baseline을 구성하는 helper입니다.
def token_signatures(tokens: TokenizedScore) -> list[tuple]:
    return [
        (
            tuple(int(value) for value in tokens.pitch[index]),
            tuple(int(value) for value in tokens.duration[index]),
        )
        for index in range(tokens.num_measures)
    ]


reference_signatures = set(token_signatures(bundle.context.tokens))


def rule_return(tokens: TokenizedScore) -> TokenizedScore:
    pitch = tokens.pitch.copy()
    duration = tokens.duration.copy()
    family = tokens.family.copy()
    event_counts = tokens.event_counts.copy()
    for source, target in true_relations.pairs:
        pitch[target] = pitch[source]
        duration[target] = duration[source]
        family[target] = family[source]
        event_counts[target] = event_counts[source]
    return replace(
        tokens,
        pitch=pitch,
        duration=duration,
        family=family,
        event_counts=event_counts,
    )


negative_relations = HwanipRelations(
    relations=(
        HwanipRelation(
            relation_id="negative_chapter_2_to_chapter_4",
            source_measures=(5, 8),
            target_measures=(25, 28),
            relation_type="return_of",
        ),
    ),
    num_measures=bundle.context.tokens.num_measures,
)

In [ ]:
# 제1장 material 생성과 제4장 conditional generation을 분리합니다.
def generate_new_chapter_one(seed: int) -> TokenizedScore:
    # 제2장 이후 악보를 고정하고 제1장 surface만 생성합니다.
    stage_one = bundle.sampler.sample(
        seed=seed,
        config=experiment_config,
        score_anchors=tuple(range(5, 73)),
        family_anchors=bundle.family_anchors,
    )
    return stage_one.tokens


def generate_return(
    initial: TokenizedScore,
    *,
    seed: int,
    relations: HwanipRelations | None,
) -> TokenizedScore:
    # 생성한 제1장과 환입 target 밖의 모든 마디를 고정합니다.
    fixed_measures = tuple([*range(1, 25), *range(29, 73)])
    original_relations = bundle.sampler.hwanip_relations
    bundle.sampler.hwanip_relations = relations
    try:
        result = bundle.sampler.sample(
            seed=seed,
            config=experiment_config,
            score_anchors=fixed_measures,
            family_anchors=bundle.family_anchors,
            initial=initial,
        )
    finally:
        bundle.sampler.hwanip_relations = original_relations
    return result.tokens

In [ ]:
# 네 조건을 동일 Seed로 실행하고 true 환입 relation에 대해 평가합니다.
rows = []
details = {}

for trial in range(N_TRIALS):
    seed = SEED_START + trial
    new_chapter_one = generate_new_chapter_one(seed)
    conditions = {
        "Proposed Graph-conditioned Model": generate_return(
            new_chapter_one, seed=seed + 100, relations=true_relations
        ),
        "No-return ablation": generate_return(
            new_chapter_one, seed=seed + 100, relations=None
        ),
        "Rule Baseline": rule_return(new_chapter_one),
        "Negative control": generate_return(
            new_chapter_one, seed=seed + 100, relations=negative_relations
        ),
    }

    for condition, tokens in conditions.items():
        signatures = token_signatures(tokens)
        rates = true_relations.counterfactual_return_rates(signatures, reference_signatures)
        row = {
            "trial": trial,
            "seed": seed,
            "condition": condition,
            **rates,
            "family satisfaction": true_relations.family_satisfaction(tokens.family),
            "surface satisfaction": true_relations.surface_satisfaction(signatures),
        }
        rows.append(row)
        details[(trial, condition)] = tokens

result_table = pd.DataFrame(rows)
display(result_table)

In [ ]:
# 조건별 평균을 계산합니다. None인 r_new/r_copy는 해당 subset이 없다는 뜻입니다.
summary_table = (
    result_table.groupby("condition", as_index=False)
    .agg(
        n_new=("n_new", "sum"),
        n_copy=("n_copy", "sum"),
        r_new=("r_new", "mean"),
        r_copy=("r_copy", "mean"),
        family_satisfaction=("family satisfaction", "mean"),
        surface_satisfaction=("surface satisfaction", "mean"),
    )
)
display(summary_table)

minimum_new_slots = 4
for _, row in summary_table.iterrows():
    if row["condition"] != "Rule Baseline" and row["n_new"] < minimum_new_slots:
        print(f"{row['condition']}: new material이 적어 r_new 해석을 보류합니다.")

## 해석 원칙

Rule Baseline은 환입을 규칙으로 복사하므로 구조적 upper bound/control이며 learned result가
아닙니다. Proposed Model의 주장은 `r_new`가 충분한 new-material sample에서 No-return
ablation과 Negative control보다 높을 때만 지지됩니다. smoke Checkpoint 또는 두 trial의
결과는 실행 예시일 뿐 논문 evidence가 아닙니다.